# 🧠 El Ciclo Básico de Deep Learning en PyTorch Puro (Sin Arnés)

> **Objetivo:** Ver y comprender todo el ciclo de entrenamiento de una red neuronal paso a paso, usando **exclusivamente PyTorch estándar** (sin librerías intermedias ni el arnés `harness.py`).

---

### Las 4 fases que se repiten en cada iteración:

```none
┌──────────────────────────────────────────────────────────┐
│ 1. PREDECIR   la red produce una salida       (forward)  │
│ 2. MEDIR      comparar con lo esperado        (pérdida)  │
│ 3. CULPAR     ¿cuánto contribuyó cada peso?   (backward) │
│ 4. CORREGIR   ajustar cada peso un poco       (update)   │
└──────────────────────────────────────────────────────────┘
```

## 0. Importaciones y configuración del entorno

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# 1. Semillas para asegurar reproducibilidad exacta
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 2. Selección de dispositivo (GPU si está disponible, sino CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

## 1. Generación y preparación de datos (`Dataset` y `DataLoader`)

Generamos un problema sintético sencillo: aproximar una función no lineal:
$$y = \sin(x) + 0.1 \cdot \epsilon$$
donde $\epsilon$ es ruido gaussiano.

In [ ]:
# 1. Crear datos sintéticos (512 muestras)
n_samples = 512
X = torch.linspace(-np.pi, np.pi, n_samples).unsqueeze(1)  # Dimensión: [512, 1]
noise = torch.randn_like(X) * 0.1                          # Ruido gaussiano
y = torch.sin(X) + noise                                   # Diana (target)

# 2. División en Entrenamiento (80%) y Validación (20%)
split_idx = int(0.8 * n_samples)
X_train, y_train = X[:split_idx], y[:split_idx]
X_val, y_val = X[split_idx:], y[split_idx:]

# 3. Envolver en TensorDataset
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

# 4. Crear los DataLoaders (lotes de 32 ejemplos)
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Muestras de entrenamiento: {len(train_dataset)} ({len(train_loader)} lotes de {BATCH_SIZE})")
print(f"Muestras de validación:    {len(val_dataset)} ({len(val_loader)} lotes de {BATCH_SIZE})")

## 2. Definición del Modelo (`nn.Module`)

Definimos un Perceptrón Multicapa (MLP) pequeño con:
- 1 entrada ($x$)
- 1 capa oculta de 16 neuronas con activación `ReLU`
- 1 salida ($y$)

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, hidden_size: int = 16):
        super().__init__()
        # Capa lineal 1: de 1 entrada a 16 características
        self.fc1 = nn.Linear(in_features=1, out_features=hidden_size)
        # Función de activación no lineal
        self.act = nn.ReLU()
        # Capa lineal 2: de 16 a 1 salida
        self.fc2 = nn.Linear(in_features=hidden_size, out_features=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Paso hacia adelante (Forward): cómo viaja la señal por la red
        h = self.act(self.fc1(x))
        out = self.fc2(h)
        return out

# Instanciar el modelo y moverlo a la memoria adecuada (CPU o GPU)
model = SimpleMLP(hidden_size=16).to(device)
print(model)

## 3. Función de Pérdida y Optimizador

- **Función de pérdida (Loss):** Error Cuadrático Medio (`MSELoss`), adecuada para regresión.
- **Optimizador:** `Adam` con tasa de aprendizaje (Learning Rate) $\eta = 0.01$ sobre los parámetros `model.parameters()`.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-2)
epochs = 30

## 4. El Bucle de Entrenamiento (Secuencial en PyTorch)

En cada época:
1. **Entrenamiento (`model.train()`):** Se itera por los lotes y se aplican las 4 fases.
2. **Validación (`model.eval()` con `@torch.no_grad()`):** Se calcula el error sobre datos nunca vistos para comprobar generalización.

In [ ]:
history = []  # Para registrar las métricas de cada época

for epoch in range(epochs):
    # =========================================================================
    # FASE A: ENTRENAMIENTO
    # =========================================================================
    model.train()  # Activar modo entrenamiento
    total_train_loss = 0.0
    total_train_samples = 0

    for inputs, targets in train_loader:
        # Enviar tensores al dispositivo correspondiente
        inputs = inputs.to(device)
        targets = targets.to(device)

        # --- PASO 0: Limpiar gradientes anteriores ---
        optimizer.zero_grad()

        # --- PASO 1: PREDECIR (Forward) ---
        predictions = model(inputs)

        # --- PASO 2: MEDIR (Pérdida / Loss) ---
        loss = criterion(predictions, targets)

        # --- PASO 3: CULPAR (Backward / Gradientes) ---
        loss.backward()

        # --- PASO 4: CORREGIR (Update de pesos) ---
        optimizer.step()

        # Acumular pérdida ponderada por el tamaño del lote
        total_train_loss += loss.item() * len(inputs)
        total_train_samples += len(inputs)

    mean_train_loss = total_train_loss / total_train_samples

    # =========================================================================
    # FASE B: EVALUACIÓN / VALIDACIÓN
    # =========================================================================
    model.eval()  # Activar modo evaluación (desactiva dropout/batchnorm)
    total_val_loss = 0.0
    total_val_samples = 0

    # Desactivar Autograd para ahorrar memoria y tiempo en validación
    with torch.no_grad():
        for val_inputs, val_targets in val_loader:
            val_inputs = val_inputs.to(device)
            val_targets = val_targets.to(device)

            # Solo forward + medir pérdida (sin backward ni step)
            val_predictions = model(val_inputs)
            val_loss = criterion(val_predictions, val_targets)

            total_val_loss += val_loss.item() * len(val_inputs)
            total_val_samples += len(val_inputs)

    mean_val_loss = total_val_loss / total_val_samples

    # =========================================================================
    # REGISTRO Y SALIDA POR CONSOLA (Idéntica al formato de harness)
    # =========================================================================
    history.append({
        "epoch": epoch,
        "train_loss": mean_train_loss,
        "val_loss": mean_val_loss
    })

    # Mostrar progreso cada 5 épocas y en la última
    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f"  epoch {epoch:3d}  train {mean_train_loss:.5f}  val {mean_val_loss:.5f}")

## 5. Visualización de Resultados

Graficamos:
1. **Curva de aprendizaje:** Evolución de la pérdida en train y val a lo largo de las épocas.
2. **Ajuste del modelo:** Comparación de los datos reales frente a la curva predicha por la red.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# 1. Curva de Pérdida (Learning Curve)
epochs_range = [h["epoch"] for h in history]
train_losses = [h["train_loss"] for h in history]
val_losses = [h["val_loss"] for h in history]

ax1.plot(epochs_range, train_losses, label="train_loss", color="tab:blue", lw=2)
ax1.plot(epochs_range, val_losses, label="val_loss", color="tab:orange", linestyle="--", lw=2)
ax1.set_yscale("log")
ax1.set_xlabel("Época")
ax1.set_ylabel("Pérdida (MSE, escala log)")
ax1.set_title("Curva de Aprendizaje")
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Ajuste de la Función Aprendida
model.eval()
with torch.no_grad():
    X_test = torch.linspace(-np.pi, np.pi, 200).unsqueeze(1).to(device)
    y_pred = model(X_test).cpu().numpy()

ax2.scatter(X.numpy(), y.numpy(), alpha=0.3, label="Datos reales (con ruido)", color="gray", s=15)
ax2.plot(X_test.cpu().numpy(), np.sin(X_test.cpu().numpy()), label="Función original sin(x)", color="green", lw=1.5, linestyle=":")
ax2.plot(X_test.cpu().numpy(), y_pred, label="Predicción del MLP", color="tab:red", lw=2.5)
ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.set_title("Ajuste del Modelo")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Persistencia (Guardar los pesos)

Guardamos los parámetros entrenados del modelo en disco usando `torch.save` y su `state_dict()`:

In [ ]:
torch.save(model.state_dict(), "modelo_simple.pt")
print("✅ Pesos del modelo guardados correctamente en 'modelo_simple.pt'")
print(f"Métrica final val_loss: {history[-1]['val_loss']:.5f}")

---

## 💡 Comparación: ¿Qué hace `harness.py` por nosotros?

| Este Notebook (PyTorch puro) | En el arnés (`harness.py`) |
|---|---|
| Definir `SimpleMLP(nn.Module)` | `@models.register("mlp")` |
| Generar tensores y `DataLoader` | `@datasets.register("sin_wave")` |
| Escribir el bucle `for epoch` manual | `H.run_experiment(config, seed=0)` |
| Gestionar `history.append(...)` | Registrado automáticamente en `result.history` y exportado a `metrics.csv` |
| `torch.save(model.state_dict(), ...)` | Guardado automáticamente en `runs/<run_id>/weights.pt` junto a `meta.json` |